# Baseline Detector Evaluation
Evaluates the single-pass LLM baseline on the test set

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from baseline_detector import BaselineDetector
from tqdm import tqdm

In [ ]:
# Load test set
test_df = pd.read_csv('test_split.csv')
print(f"Test set size: {len(test_df)}")
print(f"Label distribution: {test_df['label'].value_counts().to_dict()}")

In [ ]:
# Initialize detector
detector = BaselineDetector(model_name="gpt-3.5-turbo")

In [ ]:
# Run predictions on test set (sample first 100 for speed/cost)
sample_size = 100
test_sample = test_df.sample(n=sample_size, random_state=42)

predictions = []
confidences = []

for review in tqdm(test_sample['Review'].values, desc="Evaluating"):
    result = detector.detect(review)
    # Convert prediction to binary (0=human, 1=ai)
    pred = 1 if result.prediction == "ai" else 0
    predictions.append(pred)
    confidences.append(result.confidence)

test_sample['prediction'] = predictions
test_sample['confidence'] = confidences

In [ ]:
# Calculate metrics
y_true = test_sample['label'].values
y_pred = test_sample['prediction'].values

accuracy = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary')

print("\n=== Baseline Performance ===")
print(f"Accuracy:  {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1-Score:  {f1:.3f}")

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Human', 'AI'],
            yticklabels=['Human', 'AI'])
plt.title('Confusion Matrix - Baseline Detector')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

In [ ]:
# Confidence distribution by true label
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Human reviews
human_conf = test_sample[test_sample['label'] == 0]['confidence']
axes[0].hist(human_conf, bins=20, color='blue', alpha=0.7)
axes[0].set_title('Confidence Distribution - Human Reviews')
axes[0].set_xlabel('Confidence')
axes[0].set_ylabel('Count')

# AI reviews
ai_conf = test_sample[test_sample['label'] == 1]['confidence']
axes[1].hist(ai_conf, bins=20, color='red', alpha=0.7)
axes[1].set_title('Confidence Distribution - AI Reviews')
axes[1].set_xlabel('Confidence')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Save results
test_sample.to_csv('baseline_predictions.csv', index=False)
print("\n✓ Results saved to baseline_predictions.csv")